<a href="https://colab.research.google.com/github/Microblast56/Flyrank_ml_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Microblast56/Flyrank_ml_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / Scoring**

My lane is Refresh / Content Opportunity Scoring. The practical question is
not simply whether a page is declining; it is which pages should be reviewed
first. The output will therefore be a score that allows pages to be ranked by
priority for human review. Ranking/scoring fits the decision because an SEO or
content editor has limited time and needs an ordered list rather than only a
yes/no prediction.

## 2. Target or proxy

For the starter dataset, I will use `is_declining_proxy` as a temporary proxy
target. It is defined as `trend_direction == "down"`, meaning the page's recent
impression trend is classified as down.

This is a defined proxy from the current data window, not a future observed
outcome. It is useful for learning the workflow, but it should not be treated
as proof that a page will decline in the future. A stronger later version of
the project should define a future-window outcome and keep the feature window
separate from that target window.

In [ ]:
# State the task framing in executable form
task_type = "Ranking / Scoring"
decision = "Prioritize pages for human review"

print("Task type:", task_type)
print("Decision supported:", decision)

Task type: Ranking / Scoring
Decision supported: Prioritize pages for human review


## 3. Success metric

**Success metric: Precision@50**

Precision@50 measures the fraction of the top 50 pages in the ranked review
queue that are positive according to the target or proxy being evaluated.

This metric matches the real decision because the output is intended to help an
SEO or content editor prioritize a limited review queue. A higher Precision@50
means that more of the pages placed near the top of the queue are relevant to
the defined target.

For this stage, the target is only the starter proxy, so a good Precision@50
does not prove that the pages will decline in the future or that a refresh will
work. It only measures how well the ranking places proxy-positive pages near
the top.

In [ ]:
# Calculate Precision@50 for a simple transparent baseline.

import numpy as np

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    top_k_labels = np.asarray(labels)[order[:k]]
    return top_k_labels.mean()

# Simple baseline:
# prioritize pages that are both stale and visible.
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

baseline_score = stale * visible * df["impressions_90d"]

precision_50 = precision_at_k(
    baseline_score,
    df["is_declining_proxy"],
    50
)

print(f"Baseline Precision@50: {precision_50:.3f}")

Baseline Precision@50: 0.680


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content item (page)** in the
starter dataset. Each row represents one page and contains its content
properties and aggregated search/engagement measurements for the available
window.

For this lane, the target/proxy is also attached at the page level:
`is_declining_proxy = 1` when `trend_direction == "down"`, otherwise `0`.

The page and client IDs are used only to identify/group observations; they are
not predictive features.

In [ ]:
# Show the actual unit of analysis as a dataframe.

analysis_columns = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "days_since_last_update",
    "ctr",
    "is_declining_proxy"
]

lane_df = df[analysis_columns].copy()

print(f"Rows in lane dataframe: {len(lane_df):,}")
print("One row = one pseudonymized content page")
print("\nTarget/proxy column: is_declining_proxy")
print("0 = not proxy-positive")
print("1 = proxy-positive")

display(lane_df.head(10))

Rows in lane dataframe: 30,000
One row = one pseudonymized content page

Target/proxy column: is_declining_proxy
0 = not proxy-positive
1 = proxy-positive


,content_id,client_id,content_type,impressions_90d,days_since_last_update,ctr,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,3803,20,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,25,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,20,0.09,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,22,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,14,0.13,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,20,0.03,1
6,content_9a34b442b552,client_8722616204,keyword article,20,20,0.00,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,22,0.06,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,20,0.09,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,104,0.16,1


## 5. Why ML beats a fixed rule here

A fixed rule is useful because it is simple, transparent, and easy for an
editor to understand. However, page-priority decisions can depend on several
observable signals at the same time, including impressions, CTR, average
position, content age, freshness, sessions, engagement, and content
characteristics.

The useful case for ML is when these signals interact in ways that are
difficult to capture with a small number of hand-written thresholds. A model
may be able to combine these signals and produce a more useful ranking of
pages for human review.

This is a hypothesis to test, not an assumption. The fixed rule remains a
valid baseline, and ML should only be preferred if it improves the relevant
ranking metric under honest validation on data it has not effectively seen
before. If it does not improve the ranking, the simpler rule may be the better
choice.

In [ ]:
# Make the baseline benchmark explicit for this framing.

baseline_precision_at_50 = precision_at_k(
    baseline_score,
    df["is_declining_proxy"],
    50
)

features_to_consider = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "sessions_90d",
    "engagement_rate",
    "word_count"
]

print(f"Current baseline Precision@50: {baseline_precision_at_50:.3f}")
print(f"Candidate observable signals: {len(features_to_consider)}")
print("ML justification: test whether combinations of these signals improve")
print("ranking beyond a transparent fixed rule.")

Current baseline Precision@50: 0.680
Candidate observable signals: 9
ML justification: test whether combinations of these signals improve
ranking beyond a transparent fixed rule.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [ ✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.